# PCam tumor-patch classifier: best-results recipe
**For research and education only. Not for medical diagnosis.**

1. Runtime > Change runtime type > **T4 GPU** (Kaggle: Accelerator GPU, Internet on).
2. Run the cells in order. The recipe trains two models (ResNet18 and EfficientNet-B0), picks the better one using the **validation** split only, then evaluates that one on the test split once.
3. Rough time guess (not measured): 1-3 hours for both models plus a ~7.6 GB data download. The training cell prints real timings after epoch 1.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/NISTALTALSON/pcam-tumor-classifier
%cd pcam-tumor-classifier
!pip install -q -r requirements.txt

## Quick check (a few minutes). If this fails, stop and fix it before the long runs.

In [ ]:
!python train.py --quick-test

## Model 1: ResNet18
If the session disconnects: re-run the setup cell, then add `--resume` to the command.

In [ ]:
!python train.py --arch resnet18 --out runs/resnet18

## Model 2: EfficientNet-B0 (slower per epoch; skip this cell if you are short on time)

In [ ]:
!python train.py --arch efficientnet_b0 --out runs/efficientnet_b0

## Pick the winner by validation AUC, then test it ONCE

In [ ]:
!python compare_runs.py
BEST = open('runs/best.txt').read().strip()
!python evaluate.py --weights {BEST}
!python export.py --checkpoint {BEST}

In [ ]:
!zip -qr pcam_release.zip release space outputs runs/*/train_summary.json runs/*/history.png
try:
    from google.colab import files; files.download('pcam_release.zip')
except ImportError:
    print('Kaggle: download pcam_release.zip from the Output tab.')